# Reto Northwind 

```bash
py -m pip install pandas sqlalchemy psycopg2-binary matplotlib
```

Cargar la base de datos `northwind` en PostgreSQL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

Configuración de la conexión

In [ ]:
DB_USER = "maialen"
DB_PASSWORD = "maialen"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "northwind"

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

def sql_df(query: str) -> pd.DataFrame:
    return pd.read_sql(text(query), engine)

sql_df("SELECT version();")

## 1. Familiarizarse con la base de datos

In [ ]:
query_tablas = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""
sql_df(query_tablas)

In [ ]:
query_columnas = """
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
"""
sql_df(query_columnas)

## 2. Primeras consultas

### 2.1 ¿Cuántos empleados tenemos contratados? Indica id, nombre, apellido, ciudad y país.

In [ ]:
query_empleados = """
SELECT employee_id, first_name, last_name, city, country
FROM employees
ORDER BY employee_id;
"""
df_empleados = sql_df(query_empleados)
print(f"Total de empleados: {len(df_empleados)}")
df_empleados

### 2.2 ¿Qué productos tenemos? Indica id del producto, id del proveedor, nombre del producto, precio por unidad, unidades en stock, unidades pedidas al proveedor y descontinuados.

In [ ]:
query_productos = """
SELECT product_id, supplier_id, product_name, unit_price, units_in_stock, units_on_order, discontinued
FROM products
ORDER BY product_id;
"""
df_productos = sql_df(query_productos)
df_productos

### 2.3 ¿Tenemos productos descontinuados? Indica nombre del producto y cantidad en stock.

In [ ]:
query_descontinuados = """
SELECT product_name, units_in_stock
FROM products
WHERE discontinued = 1
ORDER BY product_name;
"""
df_descontinuados = sql_df(query_descontinuados)
print(f"Productos descontinuados: {len(df_descontinuados)}")
df_descontinuados

### 2.4 ¿Qué proveedores tenemos? Indica id de la compañía, nombre, ciudad y país.

In [ ]:
query_proveedores = """
SELECT supplier_id, company_name, city, country
FROM suppliers
ORDER BY supplier_id;
"""
df_proveedores = sql_df(query_proveedores)
df_proveedores

### 2.5 ¿Qué pedidos hemos tenido? Indica número de pedido, id del cliente, id del transportista, día del pedido, día requerido y día real de llegada.

In [ ]:
query_pedidos = """
SELECT order_id, customer_id, ship_via, order_date, required_date, shipped_date
FROM orders
ORDER BY order_id;
"""
df_pedidos = sql_df(query_pedidos)
df_pedidos.head()

### 2.6 ¿Cuántos pedidos hemos tenido?

In [ ]:
query_total_pedidos = """
SELECT COUNT(*) AS total_pedidos
FROM orders;
"""
sql_df(query_total_pedidos)

### 2.7 ¿Cuántos clientes tenemos? Indica id del cliente, nombre de la compañía, ciudad y país.

In [ ]:
query_clientes = """
SELECT customer_id, company_name, city, country
FROM customers
ORDER BY customer_id;
"""
df_clientes = sql_df(query_clientes)
print(f"Total de clientes: {len(df_clientes)}")
df_clientes.head()

### 2.8 ¿Con qué empresas de transporte trabajamos? Indica id del transportista y nombre de la compañía.

In [ ]:
query_transportistas = """
SELECT shipper_id, company_name
FROM shippers
ORDER BY shipper_id;
"""
df_transportistas = sql_df(query_transportistas)
df_transportistas

### 2.9 ¿Cómo son las relaciones de reporte de resultados entre los empleados?

In [ ]:
query_reportes = """
SELECT
    e.employee_id,
    e.first_name || ' ' || e.last_name AS empleado,
    e.reports_to AS manager_id,
    j.first_name || ' ' || j.last_name AS manager
FROM employees e
LEFT JOIN employees j
    ON e.reports_to = j.employee_id
ORDER BY e.employee_id;
"""
df_reportes = sql_df(query_reportes)
df_reportes

## 3. Análisis de la empresa

### 3.1 Crear los DataFrames pedidos-clientes y productos-proveedores-detalles

In [ ]:
query_pedidos_clientes = """
SELECT
    o.order_id,
    o.customer_id,
    c.company_name,
    c.city,
    c.country,
    o.employee_id,
    o.order_date,
    o.required_date,
    o.shipped_date,
    o.ship_via,
    o.freight
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id;
"""

query_productos_proveedores_detalles = """
SELECT
    od.order_id,
    od.product_id,
    p.product_name,
    p.category_id,
    p.supplier_id,
    s.company_name AS supplier_name,
    s.country AS supplier_country,
    od.unit_price,
    od.quantity,
    od.discount,
    p.units_in_stock,
    p.units_on_order,
    p.discontinued
FROM order_details od
LEFT JOIN products p
    ON od.product_id = p.product_id
LEFT JOIN suppliers s
    ON p.supplier_id = s.supplier_id;
"""

df_pedidos_clientes = sql_df(query_pedidos_clientes)
df_ppd = sql_df(query_productos_proveedores_detalles)

df_pedidos_clientes.head(), df_ppd.head()

### 3.2 Evolución de los pedidos a lo largo del tiempo

In [ ]:
query_evolucion_pedidos = """
SELECT
    EXTRACT(YEAR FROM order_date) AS ano,
    EXTRACT(MONTH FROM order_date) AS mes,
    COUNT(*) AS pedidos
FROM orders
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df_evolucion = sql_df(query_evolucion_pedidos)
df_evolucion["fecha"] = pd.to_datetime(
    df_evolucion["ano"].astype(int).astype(str) + "-" +
    df_evolucion["mes"].astype(int).astype(str).str.zfill(2) + "-01"
)
df_evolucion

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_evolucion["fecha"], df_evolucion["pedidos"], marker="o")
plt.title("Evolución mensual de los pedidos")
plt.xlabel("Fecha")
plt.ylabel("Número de pedidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.3 Países donde tenemos más ventas y distribución por continente

In [ ]:
ventas_pais = (
    df_pedidos_clientes.groupby("country", dropna=False)["order_id"]
    .nunique()
    .reset_index(name="num_pedidos")
    .sort_values("num_pedidos", ascending=False)
)

continentes = {
    "Europe": ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Ireland",
               "Italy", "Norway", "Poland", "Portugal", "Spain", "Sweden", "Switzerland", "UK"],
    "America": ["Argentina", "Brazil", "Canada", "Mexico", "USA", "Venezuela"]
}

def asignar_continente(pais):
    for continente, paises in continentes.items():
        if pais in paises:
            return continente
    return "Other"

ventas_pais["continent"] = ventas_pais["country"].apply(asignar_continente)
ventas_pais.head(20)

In [ ]:
ventas_continente = (
    ventas_pais.groupby("continent", dropna=False)["num_pedidos"]
    .sum()
    .reset_index()
    .sort_values("num_pedidos", ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(ventas_continente["continent"], ventas_continente["num_pedidos"])
plt.title("Distribución de pedidos por continente")
plt.xlabel("Continente")
plt.ylabel("Número de pedidos")
plt.tight_layout()
plt.show()

ventas_pais.sort_values("num_pedidos", ascending=False).head(15)

### 3.4 Retrasos en pedidos y relación con la compañía de transporte

In [ ]:
query_retrasos = """
SELECT
    o.order_id,
    s.company_name AS transportista,
    o.order_date,
    o.required_date,
    o.shipped_date,
    CASE
        WHEN o.shipped_date IS NULL THEN NULL
        ELSE (o.shipped_date - o.required_date)
    END AS dias_retraso
FROM orders o
LEFT JOIN shippers s
    ON o.ship_via = s.shipper_id;
"""
df_retrasos = sql_df(query_retrasos)
df_retrasos.head()

In [ ]:
sin_fecha_llegada = df_retrasos["shipped_date"].isna().sum()
total_pedidos = len(df_retrasos)
print(f"Pedidos sin fecha de llegada registrada: {sin_fecha_llegada} de {total_pedidos}")

In [ ]:
df_box = df_retrasos.dropna(subset=["dias_retraso"]).copy()

plt.figure(figsize=(10, 5))
df_box.boxplot(column="dias_retraso", by="transportista", grid=False)
plt.title("Días de retraso por transportista")
plt.suptitle("")
plt.xlabel("Transportista")
plt.ylabel("Días de retraso (shipped_date - required_date)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df_retrasos.groupby("transportista", dropna=False).agg(
    pedidos=("order_id", "count"),
    sin_fecha=("shipped_date", lambda s: s.isna().sum()),
    retraso_medio=("dias_retraso", "mean"),
    retraso_mediano=("dias_retraso", "median")
).sort_values("retraso_medio", ascending=False)

### 3.5 Distribución media del precio del pedido por país del cliente

In [ ]:
query_precio_pedido_pais = """
SELECT
    c.country,
    o.order_id,
    AVG(od.unit_price * od.quantity * (1 - od.discount)) AS valor_medio_lineas_pedido,
    SUM(od.unit_price * od.quantity * (1 - od.discount)) AS valor_total_pedido
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_details od
    ON o.order_id = od.order_id
GROUP BY c.country, o.order_id
ORDER BY c.country, o.order_id;
"""
df_precio_pedido_pais = sql_df(query_precio_pedido_pais)
df_precio_pedido_pais.head()

In [ ]:
resumen_precio_pais = (
    df_precio_pedido_pais.groupby("country")["valor_total_pedido"]
    .mean()
    .reset_index(name="media_valor_pedido")
    .sort_values("media_valor_pedido", ascending=False)
)

plt.figure(figsize=(12, 6))
plt.bar(resumen_precio_pais["country"], resumen_precio_pais["media_valor_pedido"])
plt.title("Valor medio del pedido por país del cliente")
plt.xlabel("País")
plt.ylabel("Media del valor total del pedido")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

resumen_precio_pais

### 3.6 Clientes que no han pedido nunca y porcentaje

In [ ]:
query_clientes_sin_pedidos = """
SELECT
    c.customer_id,
    c.company_name,
    c.country
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
"""
df_clientes_sin_pedidos = sql_df(query_clientes_sin_pedidos)

query_porcentaje_clientes_sin_pedidos = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN o.order_id IS NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS porcentaje_clientes_sin_pedidos
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id;
"""
df_porcentaje_sin_pedidos = sql_df(query_porcentaje_clientes_sin_pedidos)

df_clientes_sin_pedidos, df_porcentaje_sin_pedidos

### 3.7 Productos más demandados y necesidad de restock

In [ ]:
query_demanda_productos = """
SELECT
    p.product_id,
    p.product_name,
    SUM(od.quantity) AS unidades_demandadas,
    p.units_in_stock,
    p.units_on_order,
    p.reorder_level,
    p.discontinued
FROM products p
LEFT JOIN order_details od
    ON p.product_id = od.product_id
GROUP BY
    p.product_id, p.product_name, p.units_in_stock, p.units_on_order, p.reorder_level, p.discontinued
ORDER BY unidades_demandadas DESC NULLS LAST;
"""
df_demanda = sql_df(query_demanda_productos)
df_demanda["requiere_restock"] = (df_demanda["units_in_stock"] <= 20) & (df_demanda["units_on_order"] == 0)
df_demanda.head(20)

In [ ]:
top_demandados = df_demanda.sort_values("unidades_demandadas", ascending=False).head(15)

plt.figure(figsize=(12, 6))
plt.bar(top_demandados["product_name"], top_demandados["unidades_demandadas"])
plt.title("Top 15 productos más demandados")
plt.xlabel("Producto")
plt.ylabel("Unidades demandadas")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

df_restock = df_demanda[df_demanda["requiere_restock"]].sort_values("unidades_demandadas", ascending=False)
df_restock

## 4. Queries avanzadas

### 4.1 Última vez que se pidió un producto de cada categoría

In [ ]:
query_ultima_vez_categoria = """
SELECT
    c.category_id,
    c.category_name,
    MAX(o.order_date) AS ultima_fecha_pedido
FROM categories c
JOIN products p ON c.category_id = p.category_id
JOIN order_details od ON p.product_id = od.product_id
JOIN orders o ON od.order_id = o.order_id
GROUP BY c.category_id, c.category_name
ORDER BY c.category_id;
"""
sql_df(query_ultima_vez_categoria)

### 4.2 ¿Existe algún producto que nunca se haya vendido por su precio original?

In [ ]:
query_precio_original = """
SELECT DISTINCT
    p.product_id,
    p.product_name,
    p.unit_price AS precio_actual
FROM products p
WHERE NOT EXISTS (
    SELECT 1
    FROM order_details od
    WHERE od.product_id = p.product_id
      AND od.unit_price = p.unit_price
);
"""
sql_df(query_precio_original)

### 4.3 Productos con categoría 'Confections': ID de producto, nombre e ID de categoría

In [ ]:
query_confections = """
SELECT
    p.product_id,
    p.product_name,
    p.category_id
FROM products p
JOIN categories c ON p.category_id = c.category_id
WHERE c.category_name = 'Confections'
ORDER BY p.product_id;
"""
sql_df(query_confections)

### 4.4 Proveedores prescindibles porque todos sus productos están descontinuados

In [ ]:
query_proveedores_prescindibles = """
SELECT
    s.supplier_id,
    s.company_name
FROM suppliers s
JOIN products p ON s.supplier_id = p.supplier_id
GROUP BY s.supplier_id, s.company_name
HAVING COUNT(*) = SUM(CASE WHEN p.discontinued = 1 THEN 1 ELSE 0 END)
ORDER BY s.supplier_id;
"""
sql_df(query_proveedores_prescindibles)

### 4.5 Clientes que compraron más de 30 artículos 'Chai' en un único pedido

In [ ]:
query_clientes_chai = """
SELECT
    o.order_id,
    c.customer_id,
    c.company_name,
    p.product_name,
    od.quantity
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_details od ON o.order_id = od.order_id
JOIN products p ON od.product_id = p.product_id
WHERE p.product_name = 'Chai'
  AND od.quantity > 30
ORDER BY od.quantity DESC;
"""
sql_df(query_clientes_chai)

### 4.6 Clientes cuya suma total de carga (freight) en pedidos sea mayor de 1000

In [ ]:
query_clientes_freight = """
SELECT
    c.customer_id,
    c.company_name,
    SUM(o.freight) AS carga_total
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.company_name
HAVING SUM(o.freight) > 1000
ORDER BY carga_total DESC;
"""
sql_df(query_clientes_freight)

### 4.7 Ciudades con 5 o más empleadas

In [ ]:
query_ciudades_empleadas = """
SELECT
    city,
    COUNT(*) AS total_empleadas
FROM employees
WHERE title_of_courtesy IN ('Ms.', 'Mrs.')
GROUP BY city
HAVING COUNT(*) >= 5
ORDER BY total_empleadas DESC, city;
"""
sql_df(query_ciudades_empleadas)